### Task 1: Fine-Tuning with LoRA
Fine-tuning Qwen2.5-0.5B-Instruct for customer support message classification and structured JSON generation.

### 1. Problem Statement
The objective is to convert natural-language customer support messages into structured JSON containing intent, reason, priority, department, and order ID.

### 2. Base Model Selection
Qwen2.5-0.5B-Instruct is selected for its small size, instruction-following capabilities, and suitability for resource-efficient fine-tuning.

In [1]:
import os, re, json, gc, math, time, random, itertools
from dataclasses import dataclass, asdict, field
from collections import Counter
import numpy as np, pandas as pd, torch
import transformers, peft
from sklearn.metrics import f1_score
from transformers import (AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments,
                          BitsAndBytesConfig, set_seed)
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

assert torch.cuda.is_available(), "Enable a GPU: Runtime → Change runtime type → T4 GPU"
BF16 = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16 else torch.float16
VERSIONS = dict(torch=torch.__version__, transformers=transformers.__version__, peft=peft.__version__,
                gpu=torch.cuda.get_device_name(0))
print(VERSIONS)

{'torch': '2.14.1+cu130', 'transformers': '5.22.1', 'peft': '0.21.2', 'gpu': 'Tesla T4'}


In [2]:
@dataclass
class Cfg:
    model_name: str = "Qwen/Qwen2.5-0.5B-Instruct"
    use_qlora: bool = False
    grad_ckpt: bool = False

    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    target_modules: tuple = ("q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj")

    lr: float = 2e-4
    epochs: int = 3
    per_device_bs: int = 8
    grad_accum: int = 2
    warmup_ratio: float = 0.05
    max_len: int = 512

    n_train: int = 1200
    n_val: int = 150
    n_test_id: int = 150
    n_test_ood: int = 150
    seed: int = 42
    out_dir: str = "./support-lora"

CFG = Cfg()

In [3]:
TAXONOMY = {
 "return":            {"department":"returns",   "reasons":["damaged_product","wrong_item","changed_mind","defective_product"]},
 "refund":            {"department":"billing",   "reasons":["refund_status","missing_refund","partial_refund"]},
 "technical_support": {"department":"technical", "reasons":["login_issue","app_error","account_locked","password_reset"]},
 "order_status":      {"department":"orders",    "reasons":["tracking_request","delayed_order","delivery_update"]},
 "cancellation":      {"department":"orders",    "reasons":["cancel_before_shipping","cancel_request"]},
 "payment_issue":     {"department":"billing",   "reasons":["payment_failed","duplicate_charge","payment_method"]},
}
REASON2INTENT = {r:i for i,v in TAXONOMY.items() for r in v["reasons"]}
assert len(REASON2INTENT) == sum(len(v["reasons"]) for v in TAXONOMY.values()), "a reason is shared by two intents"
ALL_REASONS = list(REASON2INTENT)
DEPARTMENTS = sorted({v["department"] for v in TAXONOMY.values()})
PRIORITIES  = ["low","medium","high"]
KEYS = ["intent","reason","priority","department","order_id"]

HIGH = {"damaged_product","defective_product","duplicate_charge","payment_failed","account_locked","missing_refund"}
LOW  = {"changed_mind","tracking_request","delivery_update","password_reset","payment_method"}
def priority_for(reason, urgent):
    if urgent or reason in HIGH: return "high"
    return "low" if reason in LOW else "medium"


PHRASES = {
 "damaged_product": ["The item arrived broken","My package contains a damaged product","The product was cracked when I opened the box","I received a broken item","Half of what was in the parcel is smashed","It showed up in pieces"],
 "wrong_item": ["You sent me the wrong product","The item delivered is not what I ordered","I received someone else's product","This is the incorrect item","I opened the box and found a different model than the one I bought","What you shipped doesn't match my order"],
 "changed_mind": ["I no longer want this item","I changed my mind and want to send it back","Please help me return an unwanted purchase","I would like to return this, it is not for me","On second thought I don't need this anymore and want to return it","Can I send this back? I decided against keeping it"],
 "defective_product": ["The product does not work","The device is faulty","It stopped working on the first day","The item is defective","Nothing happens when I switch it on","It died after one use"],
 "refund_status": ["Where is my refund","My refund has not arrived yet","Can you check the status of my refund","How long until I get my refund","Any news on the money you are returning to me","I'd like an update on my pending reimbursement"],
 "missing_refund": ["The return was accepted but I still have no money back","My money has not been credited after the refund approval","You approved my refund but nothing reached my account","The refund was confirmed last week and the amount is still missing","The approval email said I'd be repaid, yet my balance hasn't changed","I got the confirmation but never saw the funds"],
 "partial_refund": ["I only got part of my refund","The refund amount is less than what I paid","You refunded only a portion of the price","My refund was short by a few dollars","Why was I repaid less than the total I spent","The money returned doesn't cover the full amount"],
 "login_issue": ["I cannot log into my account","The website will not let me sign in","I am unable to access my account","Sign in fails even though my details are right","Every time I enter my credentials the page just reloads","I can't get into my profile"],
 "app_error": ["The app keeps crashing","I get an error whenever I open the application","The application is not working properly","The mobile app freezes on startup","Your app shuts down by itself right after launching","Something is broken in the app, it shows an error screen"],
 "account_locked": ["My account is locked after several attempts","I am locked out of my account","The system blocked my account","My account has been suspended for too many failed tries","Access to my profile was disabled after wrong password entries","It says my account is frozen"],
 "password_reset": ["I forgot my password","I need to reset my login password","How can I create a new password","Please send me a password reset link","I don't remember my login credentials and need new ones","Help me change my password, I lost it"],
 "tracking_request": ["Please share my tracking details","How can I track my parcel","I need an update on the shipment location","Where can I find the tracking number","Can you tell me where my package is right now","Send me the courier link so I can follow my shipment"],
 "delayed_order": ["My order is late","The delivery date has passed and nothing arrived","My package has been delayed","It has been two weeks and my order still hasn't come","The shipment was supposed to be here days ago","I'm still waiting on my order, it is overdue"],
 "delivery_update": ["When will my package be delivered","Can you tell me the expected delivery date","I need an update about delivery","What is the estimated arrival of my order","Which day should I expect the courier at my door","How much longer until it reaches me"],
 "cancel_before_shipping": ["Please cancel my order before it ships","I want to stop an order that has not shipped","Cancel my purchase before dispatch","Can you halt my order before it leaves the warehouse","Don't send my order out yet, I want to cancel it","Stop the shipment, it hasn't been dispatched and I want to cancel"],
 "cancel_request": ["I would like to cancel my order","Please cancel this purchase","I no longer want the order; cancel it","Kindly cancel my recent order","Scrap my order please","I need to call off this order"],
 "payment_failed": ["My payment failed at checkout","The transaction was declined","I cannot complete payment for my order","My card keeps getting rejected","Checkout throws a payment error every time I try","The purchase won't go through when I pay"],
 "duplicate_charge": ["I was charged twice","There are two charges for one purchase","The same payment was deducted two times","My card was billed double for one order","I see the same amount taken from me twice","You collected the money two times"],
 "payment_method": ["How do I update my card details","I need to change my payment method","Can I use a different card for this purchase","I want to add a new payment option","Can I swap the card on file for another one","Where do I edit how I pay"],
}
assert set(PHRASES) == set(ALL_REASONS) and all(len(v) == 6 for v in PHRASES.values())

OPEN  = ["", "", "Hello, ", "Hi support, ", "Please help: "]
CLOSE = ["", "", " Please assist me.", " Thank you."]
URGENT = [" This is urgent.", " I need this sorted ASAP.", " Please respond immediately."]
ID_TEMPLATES = ["Order {id}: {core}.", "{core} (order {id}).", "{core}. My order number is {id}.", "Regarding {id}: {core}."]

def _render(rng, core, order_id, urgent):
    if order_id:
        text = rng.choice(ID_TEMPLATES).format(id=order_id, core=core)
    else:
        op = rng.choice(OPEN)
        c = core if (not op or core.startswith("I ") or core.startswith("I'")) else core[0].lower() + core[1:]
        text = op + c + rng.choice([".", "?"] if c.split()[0].lower() in {"where","how","when","can","why","which","what"} else ["."])
    text += rng.choice(CLOSE)
    if urgent: text += rng.choice(URGENT)
    return text

def gen_split(rng, n, phrase_idx, id_range, seen, split):
    rows, tries = [], 0
    while len(rows) < n:
        tries += 1
        assert tries < n * 200, "could not generate enough unique examples"
        reason = ALL_REASONS[len(rows) % len(ALL_REASONS)]          # balanced over reasons
        core = PHRASES[reason][rng.choice(phrase_idx)]
        order_id = f"ORD-{rng.randint(*id_range)}" if rng.random() < 0.6 else None
        urgent = rng.random() < 0.2
        text = _render(rng, core, order_id, urgent)
        if text in seen: continue
        seen.add(text)
        intent = REASON2INTENT[reason]
        rows.append({"split": split, "input": text,
                     "output": {"intent": intent, "reason": reason, "priority": priority_for(reason, urgent),
                                "department": TAXONOMY[intent]["department"], "order_id": order_id}})
    rng.shuffle(rows)
    return rows

def build_data(cfg):
    rng, seen = random.Random(cfg.seed), set()
    SEEN, OOD = [0,1,2,3], [4,5]
    TRAIN_IDS, TEST_IDS = (10000, 59999), (60000, 99999)      # disjoint ID ranges
    d = {"train": gen_split(rng, cfg.n_train, SEEN, TRAIN_IDS, seen, "train"),
         "val":   gen_split(rng, cfg.n_val,   SEEN, TRAIN_IDS, seen, "val"),
         "test_id":  gen_split(rng, cfg.n_test_id,  SEEN, TEST_IDS, seen, "test_id"),
         "test_ood": gen_split(rng, cfg.n_test_ood, OOD,  TEST_IDS, seen, "test_ood")}

    tr = {r["input"] for r in d["train"]}
    assert not tr & {r["input"] for r in d["test_id"]} and not tr & {r["input"] for r in d["test_ood"]}
    return d

DATA = build_data(CFG)
print({k: len(v) for k, v in DATA.items()})
print(pd.Series([r["output"]["reason"] for r in DATA["train"]]).value_counts().describe()[["min","max"]].to_dict())
for r in DATA["test_ood"][:3]: print(r["input"], "->", r["output"])
json.dump(DATA, open("dataset_splits.json", "w"), indent=1)

{'train': 1200, 'val': 150, 'test_id': 150, 'test_ood': 150}
{'min': 63.0, 'max': 64.0}
It died after one use. -> {'intent': 'return', 'reason': 'defective_product', 'priority': 'high', 'department': 'returns', 'order_id': None}
Regarding ORD-89142: It showed up in pieces. Please assist me. -> {'intent': 'return', 'reason': 'damaged_product', 'priority': 'high', 'department': 'returns', 'order_id': 'ORD-89142'}
Can I send this back? I decided against keeping it. My order number is ORD-94877. Please assist me. -> {'intent': 'return', 'reason': 'changed_mind', 'priority': 'low', 'department': 'returns', 'order_id': 'ORD-94877'}


In [4]:
SYSTEM_PROMPT = (
    "You convert a customer-support message into JSON. Output ONLY one JSON object, no markdown, "
    "with exactly these keys:\n"
    f"intent: one of {list(TAXONOMY)}\n"
    f"reason: one of {ALL_REASONS}\n"
    f"priority: one of {PRIORITIES} (use high if the customer says it is urgent)\n"
    f"department: one of {DEPARTMENTS}\n"
    "order_id: the order id such as ORD-12345 if it appears in the message, otherwise null"
)
dumps = lambda o: json.dumps(o, ensure_ascii=False, separators=(",", ":"))

def build_messages(text, shots=()):
    m = [{"role": "system", "content": SYSTEM_PROMPT}]
    for s in shots:
        m += [{"role": "user", "content": s["input"]}, {"role": "assistant", "content": dumps(s["output"])}]
    return m + [{"role": "user", "content": text}]

tok = AutoTokenizer.from_pretrained(CFG.model_name)
if tok.pad_token is None: tok.pad_token = tok.eos_token
tok.padding_side = "left"

def render_prompt(text, shots=()):
    return tok.apply_chat_template(build_messages(text, shots), tokenize=False, add_generation_prompt=True)

def encode(row, max_len):
    p = tok(render_prompt(row["input"]), add_special_tokens=False)["input_ids"]
    a = tok(dumps(row["output"]) + tok.eos_token, add_special_tokens=False)["input_ids"]
    assert len(p) + len(a) <= max_len, f"example longer than max_len ({len(p)+len(a)}); raise CFG.max_len"
    return {"input_ids": p + a, "labels": [-100] * len(p) + a}

def collate(batch):
    m = max(len(b["input_ids"]) for b in batch)
    pad = lambda x, v: x + [v] * (m - len(x))
    return {"input_ids":      torch.tensor([pad(b["input_ids"], tok.pad_token_id) for b in batch]),
            "attention_mask": torch.tensor([pad([1] * len(b["input_ids"]), 0) for b in batch]),
            "labels":         torch.tensor([pad(b["labels"], -100) for b in batch])}

FEWSHOT = [next(r for r in DATA["train"] if r["output"]["intent"] == i) for i in TAXONOMY]   # one per intent
ex = encode(DATA["train"][0], CFG.max_len)
print("prompt tokens:", sum(l == -100 for l in ex["labels"]), "| answer tokens:", sum(l != -100 for l in ex["labels"]))
print(tok.decode([t for t in ex["labels"] if t != -100]))

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

prompt tokens: 223 | answer tokens: 24
{"intent":"return","reason":"wrong_item","priority":"medium","department":"returns","order_id":null}<|im_end|>


In [5]:
ORDER_ID_RE = re.compile(r"^ORD-\d{5}$")
def extract_json(text):
    i = text.find("{")
    if i < 0: return None
    try: obj, _ = json.JSONDecoder().raw_decode(text[i:])
    except json.JSONDecodeError: return None
    return obj if isinstance(obj, dict) else None

def _in(v, coll): return isinstance(v, str) and v in coll
def schema_errors(o):
    if set(o) != set(KEYS): return ["bad_keys"]
    e = []
    if not _in(o["intent"], TAXONOMY):         e.append("intent_not_in_enum")
    if not _in(o["reason"], REASON2INTENT):    e.append("reason_not_in_enum")
    if not _in(o["priority"], PRIORITIES):     e.append("priority_not_in_enum")
    if not _in(o["department"], DEPARTMENTS):  e.append("department_not_in_enum")
    oid = o["order_id"]
    if oid is not None and not (isinstance(oid, str) and ORDER_ID_RE.match(oid)): e.append("bad_order_id")
    if not e and REASON2INTENT[o["reason"]] != o["intent"]: e.append("reason_intent_inconsistent")
    if not e and TAXONOMY[o["intent"]]["department"] != o["department"]: e.append("intent_department_inconsistent")
    return e

@torch.inference_mode()
def generate(model, texts, shots=(), bs=32, max_new_tokens=80):
    model.eval(); outs = []

    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    res = {}
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        enc = tok([render_prompt(texts[i], shots) for i in idx], return_tensors="pt", padding=True,
                  add_special_tokens=False).to(model.device)
        g = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                           pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
        for i, o in zip(idx, tok.batch_decode(g[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)):
            res[i] = o.strip()
    return [res[i] for i in range(len(texts))]

def categorize(rec):
    if rec["exact"]: return "correct"
    if not rec["parsable"]: return "unparsable_output"
    if rec["schema_errors"]: return "out_of_schema:" + ",".join(rec["schema_errors"])
    p, e = rec["pred"], rec["expected"]
    if p["intent"] != e["intent"]:     return "wrong_intent"
    if p["reason"] != e["reason"]:     return "wrong_reason_within_intent"
    if p["order_id"] != e["order_id"]: return "wrong_order_id"
    if p["priority"] != e["priority"]: return "wrong_priority"
    return "other"

def evaluate(model, rows, name, shots=()):
    raws = generate(model, [r["input"] for r in rows], shots)
    recs = []
    for r, raw in zip(rows, raws):
        o, e = extract_json(raw), r["output"]
        errs = schema_errors(o) if o is not None else ["unparsable"]
        rec = {"model": name, "split": r["split"], "input": r["input"], "expected": e, "raw": raw, "pred": o,
               "parsable": o is not None, "schema_errors": errs, "schema_valid": not errs,
               "exact": o is not None and o == e}
        for k in KEYS: rec["acc_" + k] = bool(o is not None and o.get(k) == e[k])
        rec["category"] = categorize(rec)
        recs.append(rec)
    return pd.DataFrame(recs)

def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"),) * 2
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n)
    a = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (c - a) / d, (c + a) / d

def summarize(df):
    out = []
    for (m, s), g in df.groupby(["model", "split"], sort=False):
        k, n = int(g["exact"].sum()), len(g)
        lo, hi = wilson(k, n)
        yp = [(p["reason"] if isinstance(p, dict) and isinstance(p.get("reason"), str) else "<invalid>") for p in g["pred"]]
        yt = [e["reason"] for e in g["expected"]]
        out.append({"model": m, "split": s, "n": n, "exact_match": k / n, "exact_95ci": f"[{lo:.2f}, {hi:.2f}]",
                    "parsable": g["parsable"].mean(), "schema_valid": g["schema_valid"].mean(),
                    **{f"acc_{c}": g[f"acc_{c}"].mean() for c in KEYS},
                    "macroF1_reason": f1_score(yt, yp, labels=ALL_REASONS, average="macro", zero_division=0)})
    return pd.DataFrame(out).round(3)

In [6]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [7]:
def _load(name, **kw):
    try:    return AutoModelForCausalLM.from_pretrained(name, dtype=DTYPE, **kw)
    except TypeError: return AutoModelForCausalLM.from_pretrained(name, torch_dtype=DTYPE, **kw)

def load_model(cfg):
    kw = {"device_map": {"": 0}}
    if cfg.use_qlora:
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)
    return _load(cfg.model_name, **kw)

def free():
    gc.collect(); torch.cuda.empty_cache()

def run_experiment(cfg, data, base_cache=None, tag="main"):
    set_seed(cfg.seed); free()
    model = load_model(cfg); model.config.use_cache = True
    tests = data["test_id"] + data["test_ood"]
    dfs = []


    key = (cfg.model_name, cfg.use_qlora)
    if base_cache is not None and key in base_cache:
        dfs += base_cache[key]
    else:
        b = [evaluate(model, tests, "base_zero_shot"), evaluate(model, tests, "base_5_shot", FEWSHOT)]
        dfs += b
        if base_cache is not None: base_cache[key] = b


    model.config.use_cache = False
    if cfg.use_qlora:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=cfg.grad_ckpt)
    elif cfg.grad_ckpt:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.enable_input_require_grads()
    model = get_peft_model(model, LoraConfig(
        r=cfg.lora_r, lora_alpha=cfg.lora_alpha, lora_dropout=cfg.lora_dropout,
        target_modules=list(cfg.target_modules), bias="none", task_type=TaskType.CAUSAL_LM))
    for p in model.parameters():                      # fp16 AMP needs fp32 trainable params
        if p.requires_grad and p.dtype != torch.float32: p.data = p.data.float()
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_all = sum(p.numel() for p in model.parameters())
    print(f"[{tag}] trainable {n_tr:,} / {n_all:,} ({100*n_tr/n_all:.2f}%)")

    tr = [encode(r, cfg.max_len) for r in data["train"]]
    va = [encode(r, cfg.max_len) for r in data["val"]]
    args = TrainingArguments(
    output_dir=f"{cfg.out_dir}-{tag}",

    num_train_epochs=cfg.epochs,

    per_device_train_batch_size=cfg.per_device_bs,
    per_device_eval_batch_size=cfg.per_device_bs,

    gradient_accumulation_steps=cfg.grad_accum,
    learning_rate=cfg.lr,

    lr_scheduler_type="cosine",
    weight_decay=0.0,

    bf16=BF16,
    fp16=not BF16,

    gradient_checkpointing=cfg.grad_ckpt,

    optim="adamw_torch",

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,

    logging_steps=20,
    report_to="none",

    seed=cfg.seed,
    data_seed=cfg.seed,

    remove_unused_columns=False
)
    trainer = Trainer(model=model, args=args, train_dataset=tr, eval_dataset=va, data_collator=collate)
    torch.cuda.reset_peak_memory_stats(); t0 = time.time()
    trainer.train()
    train_s, peak_gb = time.time() - t0, torch.cuda.max_memory_allocated() / 1e9
    trainer.save_model(f"{cfg.out_dir}-{tag}"); tok.save_pretrained(f"{cfg.out_dir}-{tag}")
    best_eval_loss = min(h["eval_loss"] for h in trainer.state.log_history if "eval_loss" in h)


    model.config.use_cache = True
    dfs.append(evaluate(model, tests, f"lora_{tag}"))
    all_df = pd.concat(dfs, ignore_index=True)
    info = dict(tag=tag, trainable_params=n_tr, total_params=n_all, train_seconds=round(train_s, 1),
                peak_train_vram_gb=round(peak_gb, 2), best_val_loss=round(best_eval_loss, 4), cfg=asdict(cfg))
    del trainer, model; free()
    return all_df, info

BASE_CACHE = {}
RESULT_DF, INFO = run_experiment(CFG, DATA, BASE_CACHE, tag="main")
TABLE = summarize(RESULT_DF)
display(TABLE)

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[main] trainable 8,798,208 / 502,830,976 (1.75%)


Epoch,Training Loss,Validation Loss
1,0.000624,0.000535


Epoch,Training Loss,Validation Loss
1,0.000624,0.000535
2,0.000097,0.000088
3,0.000081,0.000082


,model,split,n,exact_match,exact_95ci,parsable,schema_valid,acc_intent,acc_reason,acc_priority,acc_department,acc_order_id,macroF1_reason
0,base_zero_shot,test_id,150,0.027,"[0.01, 0.07]",1.0,0.153,0.460,0.240,0.440,0.353,1.000,0.255
1,base_zero_shot,test_ood,150,0.027,"[0.01, 0.07]",1.0,0.160,0.353,0.133,0.460,0.333,1.000,0.147
2,base_5_shot,test_id,150,0.153,"[0.10, 0.22]",1.0,0.287,0.427,0.653,0.480,0.653,0.873,0.647
3,base_5_shot,test_ood,150,0.127,"[0.08, 0.19]",1.0,0.347,0.360,0.400,0.580,0.593,0.933,0.380
4,lora_main,test_id,150,1.000,"[0.98, 1.00]",1.0,1.000,1.000,1.000,1.000,1.000,1.000,1.000
5,lora_main,test_ood,150,0.587,"[0.51, 0.66]",1.0,0.833,0.833,0.660,0.827,0.907,1.000,0.663


In [8]:
cols = ["model","split","n","exact_match","exact_95ci","schema_valid","acc_reason","acc_priority","acc_order_id","macroF1_reason"]
display(TABLE[cols])
TABLE.to_csv("comparison.csv", index=False)
RESULT_DF.drop(columns=["pred"]).to_csv("all_predictions.csv", index=False)
print(INFO)

,model,split,n,exact_match,exact_95ci,schema_valid,acc_reason,acc_priority,acc_order_id,macroF1_reason
0,base_zero_shot,test_id,150,0.027,"[0.01, 0.07]",0.153,0.240,0.440,1.000,0.255
1,base_zero_shot,test_ood,150,0.027,"[0.01, 0.07]",0.160,0.133,0.460,1.000,0.147
2,base_5_shot,test_id,150,0.153,"[0.10, 0.22]",0.287,0.653,0.480,0.873,0.647
3,base_5_shot,test_ood,150,0.127,"[0.08, 0.19]",0.347,0.400,0.580,0.933,0.380
4,lora_main,test_id,150,1.000,"[0.98, 1.00]",1.000,1.000,1.000,1.000,1.000
5,lora_main,test_ood,150,0.587,"[0.51, 0.66]",0.833,0.660,0.827,1.000,0.663


{'tag': 'main', 'trainable_params': 8798208, 'total_params': 502830976, 'train_seconds': 1255.8, 'peak_train_vram_gb': 10.17, 'best_val_loss': 0.0001, 'cfg': {'model_name': 'Qwen/Qwen2.5-0.5B-Instruct', 'use_qlora': False, 'grad_ckpt': False, 'lora_r': 16, 'lora_alpha': 32, 'lora_dropout': 0.05, 'target_modules': ('q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'), 'lr': 0.0002, 'epochs': 3, 'per_device_bs': 8, 'grad_accum': 2, 'warmup_ratio': 0.05, 'max_len': 512, 'n_train': 1200, 'n_val': 150, 'n_test_id': 150, 'n_test_ood': 150, 'seed': 42, 'out_dir': './support-lora'}}


In [9]:
HYPOTHESES = {
 "out_of_schema": "Free-form decoding is not constrained to the label enum: the model produces a plausible but non-existent label "
                  "(token-level near-miss of a real label). Fix: grammar/regex-constrained decoding or logit masking over the enum.",
 "wrong_reason_within_intent": "Semantically adjacent labels (e.g. missing_refund vs refund_status, cancel_request vs cancel_before_shipping) "
                  "differ by one cue (‘approved already’, ‘not shipped yet’); with only 4 seen paraphrases per label the OOD wording "
                  "doesn't contain the cue the model learned. Fix: more/diverse paraphrases, hard-negative examples.",
 "wrong_intent": "Cross-intent lexical overlap (‘money’, ‘no longer want’, ‘order’) so the surface wording matches a different intent's "
                  "training phrases; typical of paraphrase distribution shift on a 0.5B model.",
 "wrong_order_id": "Copy/transcription error on digits, or ID inserted in a template the model rarely saw; tokenisation splits digits unevenly.",
 "wrong_priority": "Priority is a two-input rule (reason + urgency cue). The model gets the reason right but misses/over-applies the urgency bump, "
                  "i.e. the rule is only approximately learned from ~20 % urgent examples.",
 "unparsable_output": "Generation truncated or malformed; check max_new_tokens and EOS handling.",
}
ft = RESULT_DF[RESULT_DF.model.str.startswith("lora")]
print("Error buckets (fine-tuned):"); display(ft.groupby(["split","category"]).size().unstack(0).fillna(0).astype(int))
bad = ft[~ft.exact]
conf = Counter((r["expected"]["reason"], (r["pred"] or {}).get("reason", "<none>")) for _, r in bad.iterrows()
               if (r["pred"] or {}).get("reason") != r["expected"]["reason"])
print("Top reason confusions (expected → predicted):", conf.most_common(6))

picked, seen_cat = [], set()
for _, r in bad.sort_values("split", ascending=False).iterrows():      # test_ood first
    base_cat = r["category"].split(":")[0]
    if base_cat not in seen_cat: picked.append(r); seen_cat.add(base_cat)
    if len(picked) == 3: break
if len(picked) < 3: picked += [r for _, r in bad.iterrows() if r["input"] not in {p["input"] for p in picked}][:3 - len(picked)]
if not picked: print("No failures in this run – enlarge/harden the test sets; do not invent failures.")
for i, r in enumerate(picked, 1):
    print(f"\nFAILURE {i}  [{r['split']}]  category = {r['category']}")
    print("Input    :", r["input"]); print("Expected :", r["expected"]); print("Predicted:", r["pred"] if r["pred"] else r["raw"])
    print("Hypothesis:", HYPOTHESES.get(r["category"].split(":")[0], "inspect manually"))
pd.DataFrame(picked).drop(columns=["pred"], errors="ignore").to_json("failure_cases.json", orient="records", indent=2)

Error buckets (fine-tuned):


split,test_id,test_ood
category,,
correct,150,88
out_of_schema:reason_intent_inconsistent,0,16
out_of_schema:reason_not_in_enum,0,9
wrong_intent,0,10
wrong_priority,0,1
wrong_reason_within_intent,0,26


Top reason confusions (expected → predicted): [(('app_error', 'damaged_product'), 6), (('defective_product', 'damaged_product'), 5), (('missing_refund', 'partial_refund'), 5), (('cancel_before_shipping', 'cancel_request'), 5), (('password_reset', 'login_issue'), 4), (('damaged_product', 'wrong_item'), 3)]

FAILURE 1  [test_ood]  category = wrong_reason_within_intent
Input    : It died after one use.
Expected : {'intent': 'return', 'reason': 'defective_product', 'priority': 'high', 'department': 'returns', 'order_id': None}
Predicted: {'intent': 'return', 'reason': 'damaged_product', 'priority': 'high', 'department': 'returns', 'order_id': None}
Hypothesis: Semantically adjacent labels (e.g. missing_refund vs refund_status, cancel_request vs cancel_before_shipping) differ by one cue (‘approved already’, ‘not shipped yet’); with only 4 seen paraphrases per label the OOD wording doesn't contain the cue the model learned. Fix: more/diverse paraphrases, hard-negative examples.

FAILURE 2  [

In [10]:
RUN_ABLATION = False
if RUN_ABLATION:
    variants = {
        "qv_r8":        dict(lora_r=8,  lora_alpha=16, target_modules=("q_proj","v_proj")),
        "all_r4":       dict(lora_r=4,  lora_alpha=8),
        "all_r16_lr1e-4": dict(lr=1e-4),
    }
    rows = []
    for name, ov in variants.items():
        df, info = run_experiment(Cfg(**{**asdict(CFG), **ov}), DATA, BASE_CACHE, tag=name)
        t = summarize(df[df.model.str.startswith("lora")]).assign(variant=name, trainable=info["trainable_params"])
        rows.append(t)
    display(pd.concat(rows + [summarize(RESULT_DF[RESULT_DF.model.str.startswith("lora")]).assign(variant="main (all_r16)", trainable=INFO["trainable_params"])])
            [["variant","split","exact_match","schema_valid","trainable"]])

In [11]:
GB = 1e9
A = dict(layers=32, h=4096, inter=11008)
DIMS = {"q_proj":(A["h"],A["h"]), "k_proj":(A["h"],A["h"]), "v_proj":(A["h"],A["h"]), "o_proj":(A["h"],A["h"]),
        "gate_proj":(A["h"],A["inter"]), "up_proj":(A["h"],A["inter"]), "down_proj":(A["inter"],A["h"])}
def lora_params(r, targets): return A["layers"] * sum(r * sum(DIMS[t]) for t in targets)

def act_gb(b=4, s=512, h=4096, a=32, L=32, vocab=32000, ckpt=True):
    tok_ = b * s
    per_layer = s * b * h * (34 + 5 * a * s / h)
    logits = tok_ * vocab * 4 * 2                       # fp32 logits + grad
    return ((tok_ * h * 2 * L + per_layer) if ckpt else per_layer * L) / GB + logits / GB

def mem(N=7e9, E=0.26e9, r=16, targets=tuple(DIMS), ckpt=True):
    Ad = lora_params(r, targets)
    full  = N * (2 + 2 + 4 + 8) / GB
    lora  = (N * 2 + Ad * 16) / GB
    qlora = ((N - E) * (4 + 0.127) / 8 + E * 2 + Ad * 16) / GB
    ac = act_gb(ckpt=ckpt)
    return dict(adapter_params_M=Ad / 1e6, activations_gb=ac,
                full_state=full, lora_state=lora, qlora_state=qlora,
                full_total=full + ac, lora_total=lora + ac, qlora_total=qlora + ac)

for ck in (True, False):
    print("grad-ckpt:", ck, {k: round(v, 2) for k, v in mem(ckpt=ck).items()})
print("q,v only, r=8 adapter params (M):", lora_params(8, ("q_proj", "v_proj")) / 1e6)

grad-ckpt: True {'adapter_params_M': 39.98, 'activations_gb': 1.51, 'full_state': 112.0, 'lora_state': 14.64, 'qlora_state': 4.64, 'full_total': 113.51, 'lora_total': 16.15, 'qlora_total': 6.15}
grad-ckpt: False {'adapter_params_M': 39.98, 'activations_gb': 15.02, 'full_state': 112.0, 'lora_state': 14.64, 'qlora_state': 4.64, 'full_total': 127.02, 'lora_total': 29.66, 'qlora_total': 19.66}
q,v only, r=8 adapter params (M): 4.194304


In [12]:
N = INFO["total_params"] - INFO["trainable_params"]; Ad = INFO["trainable_params"]
est = (N * 2 + Ad * 16) / GB if not CFG.use_qlora else None
print("measured peak training VRAM (GB):", INFO["peak_train_vram_gb"], "| weights+adapter-state estimate (GB):", est and round(est, 2),
      "\n(the gap = activations, logits, temporary buffers – the term the 7B table adds explicitly)")

measured peak training VRAM (GB): 10.17 | weights+adapter-state estimate (GB): 1.13 
(the gap = activations, logits, temporary buffers – the term the 7B table adds explicitly)


In [13]:
report = dict(task="support message → JSON {intent,reason,priority,department,order_id}",
              versions=VERSIONS, run=INFO,
              dataset_sizes={k: len(v) for k, v in DATA.items()},
              metrics=TABLE.to_dict("records"),
              error_buckets=ft.groupby(["split", "category"]).size().rename("n").reset_index().to_dict("records"))
json.dump(report, open("experiment_report.json", "w"), indent=2, default=str)
print(json.dumps(report["run"], indent=2, default=str))

{
  "tag": "main",
  "trainable_params": 8798208,
  "total_params": 502830976,
  "train_seconds": 1255.8,
  "peak_train_vram_gb": 10.17,
  "best_val_loss": 0.0001,
  "cfg": {
    "model_name": "Qwen/Qwen2.5-0.5B-Instruct",
    "use_qlora": false,
    "grad_ckpt": false,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ],
    "lr": 0.0002,
    "epochs": 3,
    "per_device_bs": 8,
    "grad_accum": 2,
    "warmup_ratio": 0.05,
    "max_len": 512,
    "n_train": 1200,
    "n_val": 150,
    "n_test_id": 150,
    "n_test_ood": 150,
    "seed": 42,
    "out_dir": "./support-lora"
  }
}
